# 135 — Localización, mapeo y SLAM

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("robotics", seed=135)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


## Solución 1 — Deriva con 2° de sesgo

Errores de rumbo acumulados por tramo: 2°, 4°, 6°, 8°. Desvíos laterales:
`10·sin2°=0.349`, `10·sin4°=0.698`, `10·sin6°=1.045`, `10·sin8°=1.392` m.
La distancia final al origen resulta ≈ 2.4 m — prácticamente el doble de los
~1.2 m del caso de 1°. Para ángulos pequeños `sin ε ≈ ε` (en radianes), así
que la deriva es aproximadamente lineal en el sesgo; la linealidad se degrada
a medida que los ángulos crecen.


In [ ]:
import math

def deriva(bias_deg):
    x, y, heading = 0.0, 0.0, 0.0
    for _ in range(4):
        x += 10*math.cos(math.radians(heading))
        y += 10*math.sin(math.radians(heading))
        heading += 90 + bias_deg  # el giro real se pasa de largo
    return math.hypot(x, y)

for b in (1, 2):
    print(f"sesgo {b} grados -> error final {deriva(b):.2f} m")


## Solución 2 — Dimensiones y coste

- (a) Con 3 landmarks: dimensión `3 + 2·3 = 9`; P es 9×9 (81 entradas).
- (b) Con 100 landmarks: dimensión `3 + 200 = 203`; P es 203×203 (41 209
  entradas).
- (c) Coste ∝ N² sobre los landmarks: `(100/3)² ≈ 1111` veces más caro (o
  comparando dimensiones totales: `(203/9)² ≈ 509×`). Cualquiera de las dos
  cuentas muestra el punto: EKF-SLAM no escala a mapas grandes, de ahí
  FastSLAM y graph-SLAM.


In [ ]:
for n in (3, 100):
    dim = 3 + 2*n
    print(f"N={n}: dim={dim}, P={dim}x{dim}={dim*dim} entradas")
print("factor de coste ~", round((203/9)**2))


## Solución 3 — Deriva simulada y corrección

Con σ=0.05 por paso, el error tras 50 pasos es una caminata aleatoria de
desviación `0.05·√50 ≈ 0.35` m (el valor exacto depende de la semilla). La
observación del landmark con σ=0.1 y peso 0.8 colapsa el error a ~0.1 m de un
solo golpe: una única referencia externa elimina casi toda la deriva
acumulada. Eso es exactamente lo que hace un cierre de lazo.


In [ ]:
import random
random.seed(135)
x_real, x_est = 0.0, 0.0
for t in range(1, 51):
    x_real += 1
    x_est += 1 + random.gauss(0, 0.05)
    if t % 10 == 0:
        print(f"t={t}: error={abs(x_est-x_real):.3f}")
z = x_real + random.gauss(0, 0.1)
x_corr = 0.2*x_est + 0.8*z
print(f"error antes={abs(x_est-x_real):.3f}, despues={abs(x_corr-x_real):.3f}")


## Solución 4 — Ambigüedad y propagación

- (a) No. La diferencia entre las distancias esperadas (10 y 10.5) es del
  orden del ruido del sensor; ambas hipótesis son plausibles — es un caso
  clásico de asociación ambigua.
- (b) Se propaga. En EKF-SLAM la matriz P correlaciona la pose con *todos* los
  landmarks: una corrección basada en una asociación falsa mueve la pose en la
  dirección equivocada, y esa pose corrompe las siguientes correcciones de los
  demás landmarks. Por eso los sistemas reales usan puertas de validación
  (test de Mahalanobis) y descartan observaciones ambiguas en vez de
  arriesgarse.
